In [ ]:
from huggingface_hub import hf_hub_download
from datasets import load_dataset

import pandas as pd
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import umap

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10
from typing import Literal, Dict, List
import torch

In [2]:
working_dir = Path("/fsx/s3/project/P240002_mRNA_LLM/supplementary/embedding_files/CDS_BART_data")
data_name = 'CDSBART_BEST_3'
assert working_dir.is_dir, f"Working directory {working_dir} does not exist."

In [3]:
#dataframe load
data_info = pd.read_csv(working_dir / f"{data_name}_df.csv")

In [4]:
data_info
print(set(data_info['y']))

{0, 1, 2, 3}


In [5]:
class VizData:
    def __init__(
        self,
        working_dir: str | Path = working_dir,
        info_df: pd.DataFrame = data_info,
        choice_for_embedding: Literal["mean", "max", "eos"] = "mean", 
        label_info: Dict = {'plant':0 ,'bacteria':1,'invertebrate':2,'fungi':3} #, 'vertebrate_other':4},
       ):
        assert working_dir.is_dir(), f"Working directory {working_dir} does not exist."

        self.info = info_df
        self.labels = info_df["y"].values
        self.label_info = label_info
        self.reverse_label_info = {v: k for k, v in label_info.items()}

        self.choice_for_embedding = choice_for_embedding

        self.umap_df = pd.DataFrame()
        self.umap_df["labels"] = self.labels

        # use user def colors
        colors = ['darkorange', 'navy', 'green', 'red'] 
        markers = ['circle', 'square', 'triangle', 'diamond']
        
        sizes = [10, 10, 12, 14]
        self.umap_df['color'] = self.umap_df['labels'].map(lambda x: colors[x])
        self.umap_df['marker'] = self.umap_df['labels'].map(lambda x: markers[x])
        self.umap_df['legend'] = self.umap_df['labels'].map(self.reverse_label_info)
        self.umap_df['size'] = self.umap_df['labels'].map(lambda x: sizes[x])
        
        self.embedding_data = torch.load(working_dir / f"emb_no_eos_{data_name}.pt") 
        self.embeddings = self.get_embedding(choice_for_embedding, self.embedding_data)

        print(f"Embedding with shape {self.embeddings.shape}.")


    @staticmethod
    def get_embedding(choice, embedding_data):
        match choice:
            case "mean":
                embeddings = embedding_data.mean(dim=1)
            case "max":
                embeddings = embedding_data.max(dim=1) 

            case _:
                raise ValueError("Invalid choice for embedding. Choose 'mean', 'max', or 'eos'.")
        return embeddings
    
viz = VizData(
    working_dir=working_dir,
    info_df=data_info,
    choice_for_embedding="mean",
)

/fsx/tmp/jhhong/ipykernel_3209432/4258366839.py:31: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.embedding_data = torch.load(working_dir / f"emb_no_eos_{data_name}.pt"

Embedding with shape torch.Size([2000, 768]).


In [6]:
output_notebook()

pca = PCA(n_components=50)
pca_embeddings = pca.fit_transform(viz.embeddings)

umap_model = umap.UMAP(
    n_neighbors=20,
    n_epochs=1500,
    min_dist=0.0,
    init='spectral',
    repulsion_strength=4,
    metric = 'manhattan',
    random_state=42,
)

# umap_embeddings = umap_model.fit_transform(viz.embeddings)
umap_embeddings = umap_model.fit_transform(pca_embeddings)
umap_df = viz.umap_df

# add umap embeddings to the dataframe
umap_df["x"] = umap_embeddings[:, 0]
umap_df["y"] = umap_embeddings[:, 1]

# Step 5: Create Bokeh plot with tooltips
source = ColumnDataSource(umap_df)

hover = HoverTool(tooltips=[
    ("Label", "@legend"),
    ('Marker', '@marker'),
    ('Size','@size'),
])

plot = figure(
    title="UMAP for RNA CodonBERT Embeddings",    
    tools=["pan,wheel_zoom,reset", hover],
    width=700, height=600
)
# 플롯의 전체 테두리 설정
plot.outline_line_color = "black"
plot.outline_line_width = 2

plot.scatter(
    'x', 'y', 
    source=source, 
    color='color',
    marker = 'marker',
    size= 'size',
    legend_field='legend',
    alpha=0.8,
    line_color = 'white',
    line_width = 1.5
)
plot.xaxis.axis_label = 'UMAP Dimension 1'
plot.yaxis.axis_label = 'UMAP Dimension 2'
plot.legend.location = 'bottom_right'

# x축과 y축의 눈금 및 숫자 제거
plot.xaxis.visible = False
plot.yaxis.visible = False
plot.xgrid.visible = False
plot.ygrid.visible = False
show(plot)
# save(plot)


Loading BokehJS ...

/fsx/home/jhhong/miniconda3/envs/rna/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
